# Fisheye Tomography Transform

In [12]:
import os

import cv2

import mon
from fisheye import iFishTransform

# Arguments
# image_stem = "0000135_01327_d_0000151"
image_stem   = "0000305_00001_d_0000213"
distortion   = 1
area_thres   = 32
aspect_thres = 0.1

# Directories and files
current_dir     = mon.Path(os.getcwd())
root_dir        = current_dir.parents[0]
data_dir        = root_dir / "data" / "ftt"
image_file      = data_dir / f"{image_stem}.jpg"
label_file      = data_dir / f"{image_stem}.txt"
classes_file    = data_dir / f"classes.yaml"
# fft_image_file  = data_dir / f"{image_stem}_ifish.jpg"
# fft_label_file  = data_dir / f"{image_stem}_ifish.txt"
# fft_visual_file = data_dir / f"{image_stem}_ifish_viz.jpg"

# Load data
image   = cv2.imread(str(image_file))
h, w, _ = image.shape
bs      = mon.hbb.load(path=label_file, fmt=mon.BBoxFormat.YOLO, imgsz=(h, w))

classes = mon.load_config(classes_file, verbose=False)
classes = classes.get("classes", [])

# Preprocessing
sis, sbs = mon.hbb.split(image, bs, 2)

# Transform
iFish = iFishTransform(distortion, area_thres, aspect_thres)

for i, (si, sb) in enumerate(zip(sis, sbs)):
    transformed = iFish(image=si, bboxes=sb)
    fft_image   = transformed["image"]
    fft_bboxes  = transformed["bboxes"]

    # Visualize
    fft_bboxes_voc = mon.hbb.convert(fft_bboxes, fmt=mon.BBoxFormat.YOLO2VOC, imgsz=fft_image.shape[0:2])
    fft_viz = fft_image.copy()
    for j, b in enumerate(fft_bboxes_voc):
        if len(b) >= 6:
            l = f"{j} {int(b[4])}: {b[5]:.4f}"
        else:
            l = f"{j} {int(b[4])}"
        fft_viz = mon.dtypes.draw_bbox(
            image     = fft_viz,
            bbox      = b,
            label     = "",
            color     = classes[int(b[4])]["color"],
            thickness = 2,
            fill      = False,
        )

    # Save
    # fft_image_file = data_dir / f"{image_stem}_ifish_{i}.jpg"
    # fft_image_file.parent.mkdir(exist_ok=True, parents=True)
    # cv2.imwrite(str(fft_image_file),  fft_image)

    fft_visual_file = data_dir / f"{image_stem}_ifish_{i}_viz.jpg"
    fft_visual_file.parent.mkdir(exist_ok=True, parents=True)
    cv2.imwrite(str(fft_visual_file), fft_viz)

    fft_label_file  = data_dir / f"{image_stem}_ifish_{i}.txt"
    fft_label_file.parent.mkdir(exist_ok=True, parents=True)
    with open(fft_label_file, "w") as f:
        for b in fft_bboxes:
            f.write(f"{int(b[4])} {b[0]:.32f} {b[1]:.32f} {b[2]:.32f} {b[3]:.32f}\n")